In [1]:
import pandas as pd

In [2]:
%%capture
!pip install -U dspy pydantic dspy-ai

In [3]:
import dspy

api_key = ""
api_endpoint = "https://<endpoint>.services.ai.azure.com/"

lm = dspy.LM('azure/gpt-4.1', api_key = api_key, api_base=api_endpoint, api_version = '2024-10-21', max_tokens=4000)
dspy.configure(lm=lm)

In [4]:
from typing import List
from typing import Literal
from pydantic import BaseModel

class PatientRecord(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all tokens of toxic habits, if any.
    """
    
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    tobacco_habits: list[str] = dspy.OutputField(desc="All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.")
    cannabis_habits: list[str] = dspy.OutputField(desc="All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.")
    alcohol_habits: list[str] = dspy.OutputField(desc="All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.")
    drug_habits: list[str] = dspy.OutputField(desc="All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.")

term_extractor_patient = dspy.ChainOfThought(PatientRecord)

In [24]:
text = """
Mujer de casi 32 años, natural y residente en la zona, en seguimiento en nuestra UCA de Vinaròs desde los 18 años (en 2005); en terapia conmigo desde 2014 (año de mi incorporación a la plaza).
De acuerdo a las notas de la historia clínica de papel y diferentes documentos consultados para la sesión clínica -a menudo desordenados, informes de distinta procedencia y cotejo con apuntes propios-, la paciente se inició en el consumo de tabaco y alcohol a los 12 años, en el cannabis a los 13 (diario desde los 15), en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol), y años más tarde consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox.
Lena dejó los estudios con 16 años, en 2º de la ESO, optando luego por trabajos muy precarios, erráticos, y sobre todo actividades marginales, entre ellas la prostitución hace 2 o 3 años, en Valencia.
Tiene reconocida una PNC (pensión no contributiva) por discapacidad del 73%, de la cual ella siempre se ha administrado el dinero, y hace un año fue inscrita por los Servicios Sociales de la localidad en un curso remunerado de administrativo, donde las condiciones de asistencia eran relativamente exigentes (horarios madrugadores, puntualidad, presencia, exposiciones), y a Lena le costaba bastante cumplir.
Mostraba esfuerzo, también motivada por el incentivo económico, pero había días que no acudía a clase, y ésta era también una de las razones para asistir con más frecuencia y compromiso a las citas médicas y psicológicas, que justificaban la ausencia de clase ese día.
Posteriormente ingresó voluntariamente en un centro de día para rehabilitación de tóxicos, con horarios poco compatibles con el curso de administrativo, sin perder la plaza gracias a una ILT (baja médica).
Respecto al entorno de Lena, la familia se caracteriza por su carencia de estructura.
Los padres de la paciente eran toxicómanos antes y durante su infancia, y ambos ya están fallecidos.
Ella fue acogida por la abuela materna, que también ha sido la persona que ha criado a una hermana 14 años menor, de un padre diferente (éste se encuentra vivo, también era toxicómano, fue presidiario, en la actualidad visita ocasionalmente a la familia, sobre todo a su hija, la hermana de Lena que ahora tiene 18 años).
Por tanto, desde el nacimiento la tutela de la paciente ha estado con los abuelos maternos, que actualmente cuentan con 68 años la mujer y 64 el marido (este hombre puede no ser el abuelo biológico de Lena, según una referencia de un informe de la historia clínica, y es la persona sobre la que actualmente ella deposita más hostilidad y rechazo).
La abuela es quien siempre se encarga de acompañar a Lena a los dispositivos, la ha rescatado en numerosas ocasiones de sitios hostiles y caóticos, ha supervisado muchas veces tratamientos y medicaciones, gestionado citas y visitas… es la principal figura de apego de la paciente, sin duda.
"""
term_extractor_patient(patient_discharge_summary=text)

Prediction(
    reasoning='The discharge summary provides a detailed history of substance use. The patient started "consumo de tabaco y alcohol" at age 12, "en el cannabis a los 13 (diario desde los 15)", and "en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol)", and later "consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox." These are explicit mentions of tobacco, cannabis, alcohol, and various drugs (cocaína, speed, anfetaminas, éxtasis, heroína, drogas). The summary also mentions "rehabilitación de tóxicos" and "toxicómanos" in reference to the patient and her family, which are relevant drug-related terms.',
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'drogas', 'tóxicos', 'toxicómanos']
)

In [6]:
lm.history

[{'prompt': None,
  'messages': [{'role': 'system',
    'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n

In [7]:
class SplitExtract(dspy.Module):
    def __init__(self):
        self.term_extractor_patient = dspy.ChainOfThought(PatientRecord)

    def forward(self, patient_discharge_summary: str):
        texts = patient_discharge_summary.split('\n\n')
        tobacco_habits, cannabis_habits, alcohol_habits, drug_habits = [], [], [], []
        for text in texts:
            if len(text.strip()) == 0:
                continue
            prediction = self.term_extractor_patient(patient_discharge_summary=text)
            tobacco_habits.extend(prediction.tobacco_habits)
            cannabis_habits.extend(prediction.cannabis_habits)
            alcohol_habits.extend(prediction.alcohol_habits)
            drug_habits.extend(prediction.drug_habits)

        return dspy.Prediction(tobacco_habits=tobacco_habits, cannabis_habits=cannabis_habits, alcohol_habits=alcohol_habits, drug_habits=drug_habits)

In [8]:
doc_extractor = SplitExtract()
doc_extractor(patient_discharge_summary=text)

Prediction(
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'drogas', 'tóxicos', 'toxicómanos']
)

In [9]:
lm.history

[{'prompt': None,
  'messages': [{'role': 'system',
    'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n

In [ ]:
result_filename = 'pred_test_gpt41_lists_examples_per_field_temp0_fewshot_5'

In [10]:
df_records = pd.read_csv('final_test_dataset.tsv', sep='\t')

In [11]:
df_records.head()

,filename,text,trigger_annotations,attr_annotations
0,casos_clinicos_cardiologia117,Varón de 70 años. Antecedente de tuberculosis ...,[],[]
1,casos_clinicos_cardiologia483,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...",[],[]
2,casos_clinicos_habtox1,"Dolor mamario y náuseas matutinas, refiere tes...",[],[]
3,casos_clinicos_habtox11,"Se presenta el caso de un hombre de 37 años, c...",[],[]
4,casos_clinicos_habtox114,"Varón de 40 años, procedente de la Amazonía pe...",[],[]


In [12]:
from ast import literal_eval
df_records['trigger_annotations'] = df_records['trigger_annotations'].apply(literal_eval)
df_records['attr_annotations'] = df_records['attr_annotations'].apply(literal_eval)

In [13]:
df_records['response'] = ''
df_records['entities'] = ''
df_records['is_train'] = False

In [14]:
df_records_dev = df_records[~df_records['is_train']]
df_records_dev.shape

(300, 7)

In [15]:
def split_text(filename, summary_text, annotations):
    texts = summary_text.split('\n\n')
    annotations_split = []
    current_start_idx, current_end_idx = 0, 0
    start_idx = []
    for text in texts:
        current_start_idx += summary_text[current_start_idx:].index(text)
        current_end_idx = current_start_idx + len(text)
        start_idx.append(current_start_idx)
        # filter annotations between start/end
        current_annotations = []
        for ann in annotations:
            if int(ann['off0']) >= current_start_idx and int(ann['off1']) <= current_end_idx:
                current_annotations.append({
                    'filename': filename,
                    'mark': 'TOX',
                    'label': ann['label'],
                    'off0': int(ann['off0']) - current_start_idx,
                    'off1': int(ann['off1']) - current_start_idx,
                    'span': ann['span'],
                })

        annotations_split.append(current_annotations)
        
    return texts, annotations_split, start_idx

In [16]:
df_train_records = pd.read_csv('final_train_dataset.tsv', sep='\t')
df_train_records['trigger_annotations'] = df_train_records['trigger_annotations'].apply(literal_eval)
df_train_records['attr_annotations'] = df_train_records['attr_annotations'].apply(literal_eval)
df_train_records.head()

,filename,text,trigger_annotations,attr_annotations,is_train
0,32073161_ES,"El 21 de enero de 2020, ingresó en el Hospital...","[{'label': 'Tobacco', 'off0': '569', 'off1': '...","[{'label': 'Duration', 'off0': '577', 'off1': ...",True
1,32277408_ES,﻿Un hombre de 63 años ingresó en el hospital a...,"[{'label': 'Tobacco', 'off0': '274', 'off1': '...",[],True
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False
3,32426200_ES,"Una mujer de 31 años, por lo demás sana, acudi...","[{'label': 'Alcohol', 'off0': '278', 'off1': '...","[{'label': 'Type', 'off0': '299', 'off1': '318...",True
4,32586958_ES,﻿Hombre de 21 años que acudió al servicio de u...,"[{'label': 'Tobacco', 'off0': '277', 'off1': '...",[],True


In [17]:
dev_dataset_text = []
train_dataset_text = []
for index, row in df_train_records.iterrows():
    filename = row['filename']
    #if row['is_train']:
    texts, annotations, start_idx = split_text(filename, row['text'], row['trigger_annotations'])     
    filenames = [f"{filename}_{start_id}" for start_id in start_idx]
    #else:
    #    filenames = [filename]
    #    annotations = [row['trigger_annotations']]
    #    texts = [row['text']]
    #    start_idx = [0]
        
    for (text, current_annotations, filename) in zip(texts, annotations, filenames):        
        items_list_tobacco = [ label['span'] for label in current_annotations if label['label'] == 'Tobacco']
        items_list_cannabis = [ label['span'] for label in current_annotations if label['label'] == 'Cannabis']
        items_list_alcohol = [ label['span'] for label in current_annotations if label['label'] == 'Alcohol']
        items_list_drug = [ label['span'] for label in current_annotations if label['label'] == 'Drug']
    
        example = dspy.Example(patient_discharge_summary=text,
                    filename=filename,
                    annotations=current_annotations,
                    tobacco_habits=items_list_tobacco,
                    cannabis_habits=items_list_cannabis,
                    alcohol_habits=items_list_alcohol,
                    drug_habits=items_list_drug).with_inputs("patient_discharge_summary")
        
        if row['is_train']:        
            if len(annotations) > 0:
                train_dataset_text.append(example)
        else:
            dev_dataset_text.append(example)

len(dev_dataset_text)

1735

In [18]:
len(train_dataset_text)

5575

In [ ]:
# module to split and combine results

In [ ]:
import warnings
def calculate_metrics(gs, pred, subtask=['ner','norm']):
    '''       
    Calculate task Coding metrics:
    
    Two type of metrics are calculated: per document and micro-average.
    It is assumed there are not completely overlapping annotations.
    
    Parameters
    ---------- 
    gs : pandas dataframe
        with the Gold Standard. Columns are those defined in main function.
    pred : pandas dataframe
        with the predictions. Columns are those defined in main function.
    subtask : str
        subtask name
    
    Returns
    -------
    P_per_cc : pandas series
        Precision per clinical case (index contains clinical case names)
    P : float
        Micro-average precision
    R_per_cc : pandas series
        Recall per clinical case (index contains clinical case names)
    R : float
        Micro-average recall
    F1_per_cc : pandas series
        F-score per clinical case (index contains clinical case names)
    F1 : float
        Micro-average F1-score
    '''
    
    # Predicted Positives:
    Pred_Pos_per_cc = \
        pred.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    Pred_Pos = pred.drop_duplicates(subset=['filename', "offset"]).shape[0]

    # Gold Standard Positives:
    GS_Pos_per_cc = \
        gs.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    GS_Pos = gs.drop_duplicates(subset=['filename', "offset"]).shape[0]
    
    # Eliminate predictions not in GS (prediction needs to be in same clinical
    # case and to have the exact same offset to be considered valid!!!!)
    df_sel = pd.merge(pred, gs, 
                      how="right",
                      on=["filename", "offset", "label"])
    
    if subtask=='norm':
        # Check if codes are equal
        df_sel["is_valid"] = \
            df_sel.apply(lambda x: (x["code_x"] == x["code_y"]), axis=1)
    elif subtask=='ner':
        is_valid = df_sel.apply(lambda x: x.isnull().any()==False, axis=1)
        df_sel = df_sel.assign(is_valid=is_valid.values)
    else:
        raise Exception('Error! Subtask name not properly set up')

        
    # True Positives:
    TP_per_cc = (df_sel[df_sel["is_valid"] == True]
                 .groupby("filename")["is_valid"].count())
    TP = df_sel[df_sel["is_valid"] == True].shape[0]
    
    # Add entries for clinical cases that are not in predictions but are present
    # in the GS
    cc_not_predicted = (pred.drop_duplicates(subset=["filename"])
                        .merge(gs.drop_duplicates(subset=["filename"]), 
                              on='filename',
                              how='right', indicator=True)
                        .query('_merge == "right_only"')
                        .drop('_merge', axis=1))['filename'].to_list()
    for cc in cc_not_predicted:
        TP_per_cc[cc] = 0
    
    # Remove entries for clinical cases that are not in GS but are present
    # in the predictions
    cc_not_GS = (gs.drop_duplicates(subset=["filename"])
                .merge(pred.drop_duplicates(subset=["filename"]), 
                      on='filename',
                      how='right', indicator=True)
                .query('_merge == "right_only"')
                .drop('_merge', axis=1))['filename'].to_list()
    Pred_Pos_per_cc = Pred_Pos_per_cc.drop(cc_not_GS)

    # Calculate Final Metrics:
    P_per_cc =  TP_per_cc / Pred_Pos_per_cc 
    P = TP / Pred_Pos if Pred_Pos > 0 else 0
    R_per_cc = TP_per_cc / GS_Pos_per_cc
    R = TP / GS_Pos if GS_Pos > 0 else 0
    F1_per_cc = (2 * P_per_cc * R_per_cc) / (P_per_cc + R_per_cc)
    if (P+R) == 0:
        F1 = 0
        #warnings.warn('Global F1 score automatically set to zero to avoid division by zero')
        return P_per_cc, P, R_per_cc, R, F1_per_cc, F1
    F1 = (2 * P * R) / (P + R)
    
    if ((any([F1, P, R]) > 1) | any(F1_per_cc>1) | any(P_per_cc>1) | any(R_per_cc>1) ):
        warnings.warn('Metric greater than 1! You have encountered an undetected bug, please, contact antonio.miranda@bsc.es!')
                                            
    return P_per_cc, P, R_per_cc, R, F1_per_cc, F1

In [ ]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [ ]:
import numpy as np

def get_spans(example, pred):
    filename = example['filename']
    text = example['patient_discharge_summary']
    gold_annotations = example['annotations']
    
    predicted_entities = []
    predicted_spans = []
    for habit in pred.tobacco_habits:
        predicted_entities.append({
            'trigger_type': 'Tobacco',
            'trigger_text': habit
        })

    for habit in pred.alcohol_habits:
        predicted_entities.append({
            'trigger_type': 'Alcohol',
            'trigger_text': habit
        })

    for habit in pred.cannabis_habits:
        predicted_entities.append({
            'trigger_type': 'Cannabis',
            'trigger_text': habit
        })

    for habit in pred.drug_habits:
        predicted_entities.append({
            'trigger_type': 'Drug',
            'trigger_text': habit
        })

    for ent in predicted_entities:
        spans = get_entities(filename, text, ent['trigger_text'], ent['trigger_type'])
        predicted_spans.extend(spans)

    gold_spans = []

    for ent in gold_annotations:
        gold_spans.append({
            'filename': filename,
            'mark': 'TOX',
            'label': ent['label'],
            'off0': ent['off0'],
            'off1': ent['off1'],
            'span': ent['span'],
        })
    return gold_spans, predicted_spans

def f1(example, pred, trace=None): #entities only   
    gold_spans, predicted_spans = get_spans(example, pred)
    
    #  true positives, false positives and false negatives == 0
    #if len(gold_spans) == 0 and len(predicted_spans) == 0:
    #    return 1
    gs = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    pred = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])

    gs['offset'] = gs['off0'].astype(str) + ' ' + gs['off1'].astype(str)
    pred['offset'] = pred['off0'].astype(str) + ' ' + pred['off1'].astype(str)

    # prefer shorter spans, remove overlapping spans?
    gs = gs.sort_values(by=['filename', 'off0', 'off1']).drop_duplicates(subset=['filename', 'label', 'offset']).copy()
    pred = pred.sort_values(by=['filename', 'off0', 'off1']).drop_duplicates(subset=['filename', 'label', 'offset']).copy()
    P_per_cc, P, R_per_cc, R, F1_per_cc, F1 = calculate_metrics(gs, pred, subtask='ner')
       
    return F1

In [ ]:
evaluate = dspy.Evaluate(devset=dev_dataset_text, metric=f1, num_threads=16, display_progress=True, display_table=5, provide_traceback=True)
evaluate(doc_extractor)

In [19]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=5)
optimized_fewshot = labeled_fewshot_optimizer.compile(student = doc_extractor, trainset=train_dataset_text)

In [20]:
optimized_fewshot

term_extractor_patient.predict = Predict(StringSignature(patient_discharge_summary -> reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug_habits
    instructions='You are an expert in clinical NLP in Spanish. \nExtract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.\nMake sure to retrieve all tokens of toxic habits, if any.'
    patient_discharge_summary = Field(annotation=str required=True json_schema_extra={'desc': 'Patient discharge summary', '__dspy_field_type': 'input', 'prefix': 'Patient Discharge Summary:'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    tobacco_habits = Field(annotation=list[str] required=True json_schema_extra={'desc': 'All tobacco tokens that can be extracted from the discharge summary. For example: ci

In [ ]:
evaluate(optimized_fewshot) #61.1% - labels 5, 61.39 - 7

In [25]:
optimized_fewshot(patient_discharge_summary=text)

Prediction(
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína']
)

In [26]:
lm.history[-1]

{'prompt': None,
 'messages': [{'role': 'system',
   'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n[[ 

In [ ]:
embed_api_key = ""
embed_api_endpoint = "https://<endpoint>.openai.azure.com/"

embedder = dspy.Embedder('azure/text-embedding-3-large', dimensions=512, api_key = embed_api_key, api_base=embed_api_endpoint)

In [ ]:
from dspy import KNNFewShot

knn_labeled_fewshot_optimizer = KNNFewShot(k=5, trainset=train_dataset_text, vectorizer=embedder)
knn_optimized_fewshot = knn_labeled_fewshot_optimizer.compile(doc_extractor)

In [ ]:
evaluate(knn_optimized_fewshot) #50.5% after 34% data

In [29]:
from tqdm import tqdm, tqdm_notebook

for index, row in tqdm(df_records_dev.iterrows(), total=df_records_dev.shape[0]):
    if row['response'] != '':
        continue
    responses = []
    entities = []
    texts = row['text'].split('\n\n') #[row['text']] # 
    for text in texts:
        #response = optimized_fewshot(patient_discharge_summary=text) 
        #response = knn_optimized_fewshot(patient_discharge_summary=text) 
        response = optimized_fewshot(patient_discharge_summary=text) 
        responses.append(response)
        current_entities = []
        for habit in response.tobacco_habits:
            current_entities.append({
                'trigger_type': 'Tobacco',
                'trigger_text': habit
            })
        for habit in response.alcohol_habits:
            current_entities.append({
                'trigger_type': 'Alcohol',
                'trigger_text': habit
            })
        for habit in response.cannabis_habits:
            current_entities.append({
                'trigger_type': 'Cannabis',
                'trigger_text': habit
            })
        for habit in response.drug_habits:
            current_entities.append({
                'trigger_type': 'Drug',
                'trigger_text': habit
            })
        entities.extend(current_entities)
    df_records_dev.at[index, 'response'] = responses
    df_records_dev.at[index, 'entities'] = entities

100%|█████████████████████████████████████████████████████████████████████████| 300/300 [59:09<00:00, 11.83s/it]


In [30]:
df_records_dev.head()

,filename,text,trigger_annotations,attr_annotations,response,entities,is_train
0,casos_clinicos_cardiologia117,Varón de 70 años. Antecedente de tuberculosis ...,[],[],"[[tobacco_habits, cannabis_habits, alcohol_hab...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
1,casos_clinicos_cardiologia483,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN ...",[],[],"[[tobacco_habits, cannabis_habits, alcohol_hab...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
2,casos_clinicos_habtox1,"Dolor mamario y náuseas matutinas, refiere tes...",[],[],"[[tobacco_habits, cannabis_habits, alcohol_hab...","[{'trigger_type': 'Tobacco', 'trigger_text': '...",False
3,casos_clinicos_habtox11,"Se presenta el caso de un hombre de 37 años, c...",[],[],"[[tobacco_habits, cannabis_habits, alcohol_hab...","[{'trigger_type': 'Alcohol', 'trigger_text': '...",False
4,casos_clinicos_habtox114,"Varón de 40 años, procedente de la Amazonía pe...",[],[],"[[tobacco_habits, cannabis_habits, alcohol_hab...","[{'trigger_type': 'Alcohol', 'trigger_text': '...",False


In [31]:
df_records_dev.iloc[1]['entities']

[{'trigger_type': 'Tobacco', 'trigger_text': 'fumadora'}]

In [32]:
result_filename

'pred_test_gpt41_lists_examples_per_field_temp0_fewshot_5'

In [33]:
df_records_dev.to_csv(f'{result_filename}.tsv', sep='\t', index=False)

In [34]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [35]:
import re

entities_list = []

for index, row in df_records_dev.iterrows():
    if row['response'] == '':
        continue
        
    entities = row['entities']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        term = ent[f'trigger_text']
        label = ent['trigger_type']

        entity_list.extend(get_entities(row['filename'], text, term, label))        
        
    entities_list.extend(entity_list)    

In [36]:
import pandas as pd

df_entities_list = pd.DataFrame.from_records(entities_list)
df_entities_list.drop_duplicates(inplace=True)
df_entities_list.head()

,filename,mark,label,off0,off1,span
0,casos_clinicos_cardiologia117,TOX,Tobacco,342,351,Exfumador
1,casos_clinicos_cardiologia483,TOX,Tobacco,80,88,fumadora
2,casos_clinicos_habtox1,TOX,Tobacco,3208,3214,Tabaco
3,casos_clinicos_habtox1,TOX,Alcohol,3216,3223,alcohol
4,casos_clinicos_habtox1,TOX,Alcohol,2927,2935,enolismo


In [37]:
#filename, label, off0, off1, span
df_entities_list[['filename','label','off0','off1','span']].to_csv(f'{result_filename}_entities.tsv', sep='\t', index=False)
df_entities_list.shape

(2301, 6)

In [38]:
result_filename

'pred_test_gpt41_lists_examples_per_field_temp0_fewshot_5'